In [0]:
from pathlib import Path
import pandas as pd
import numpy as np

In [0]:
%sql
DROP DATABASE IF EXISTS workspace.silver CASCADE; 

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.silver
COMMENT 'Capa Silver procesados'


In [0]:
df_movies = spark.table("workspace.bronze.movies")
df_credits = spark.table("workspace.bronze.credits")

In [0]:
from pyspark.sql.functions import col, sum

nulls = df_movies.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_movies.columns
])

display(nulls)

In [0]:
from pyspark.sql.functions import col, sum

nulls = df_credits.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_credits.columns
])

display(nulls)

In [0]:
from pyspark.sql.functions import col, to_date

df_movies_clean = df_movies.select(
    col("id").alias("movie_id"),
    col("title"),
    col("original_title"),
    col("original_language").alias("language"),
    col("overview"),
    col("popularity"),
    col("vote_average"),
    col("vote_count"),
    col("adult"),
    col("video"),
    to_date(col("release_date")).alias("release_date"),
    col("runtime"),
    col("budget"),
    col("revenue")
)

In [0]:
display(df_movies_clean)

In [0]:
#df_movies_clean = df_movies_clean.dropDuplicates(["movie_id"])

In [0]:
from pyspark.sql.functions import explode

df_genres = df_movies.select(
    col("id").alias("movie_id"),
    explode("genres").alias("genre")
)

df_genres = df_genres.select(
    "movie_id",
    col("genre.id").alias("genre_id"),
    col("genre.name").alias("genre_name")
)

In [0]:
df_actors = df_credits.select(
    col("id").alias("movie_id"),
    explode("cast").alias("actor")
)

df_actors = df_actors.select(
    "movie_id",
    col("actor.id").alias("actor_id"),
    col("actor.name").alias("actor_name")
)

In [0]:
df_movies_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.movies")

In [0]:
df_genres.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.movie_genres")

In [0]:
df_actors.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.movie_actors")

---------- FIN ----------